<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/RAG/GraphRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

# Graph RAG

In [ ]:
!pip -q install networkx transformers accelerate sentence-transformers torch

In [ ]:
import networkx as nx
import numpy as np
import json
import re

from pprint import pprint
from transformers import pipeline
from sentence_transformers import SentenceTransformer

## Knowledge Base
> Documents | Knowledge Graph | Databases

In [ ]:
# Replace this by connecting with Database and Query the DB
PRODUCTS = {

    "Nike Air Max": {
        "brand": "Nike",
        "category": "Running Shoes",
        "price": 180,
        "durability": 7,
        "comfort": 9,
        "inventory": 14
    },
    "Adidas Ultraboost": {"brand": "Adidas","category": "Running Shoes","price": 160,"durability": 8,"comfort": 9,"inventory": 20},
    "ASICS Gel Nimbus": {"brand": "ASICS","category": "Running Shoes","price": 140,"durability": 9,"comfort": 8,"inventory": 11},
    "Puma Velocity Nitro": {"brand": "Puma","category": "Running Shoes","price": 120,"durability": 8,"comfort": 8,"inventory": 18},
    "New Balance 1080": {"brand": "New Balance","category": "Running Shoes","price": 150,"durability": 8,"comfort": 9,"inventory": 7},
    "Reebok Floatride": {"brand": "Reebok","category": "Running Shoes","price": 100,"durability": 8,"comfort": 7,"inventory": 25}
}

SIMILARITIES = [

    ("Nike Air Max", "Adidas Ultraboost"),
    ("Nike Air Max", "ASICS Gel Nimbus"),
    ("Nike Air Max", "Puma Velocity Nitro"),
    ("Nike Air Max", "New Balance 1080"),
    ("Nike Air Max", "Reebok Floatride"),

    ("Adidas Ultraboost", "ASICS Gel Nimbus"),
    ("Puma Velocity Nitro", "Reebok Floatride")
]

In [ ]:
# Replace this cosntruction step by connecting with Database and Query the DB .This is done for academic training purposes
import matplotlib.pyplot as plt
G = nx.Graph()
for product, attrs in PRODUCTS.items():
    G.add_node(product,node_type="product",**attrs)
for a, b in SIMILARITIES:
    G.add_edge(a,b,relation="similar_to")

# Plot
plt.figure(figsize=(14, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G,pos,with_labels=True,node_color="skyblue",node_size=4000,font_size=9,font_weight="bold",edge_color="gray",arrows=True)
plt.title("NLP / RAG Knowledge Graph")
plt.show()

## Ingestion & Preprocessor
> Natural Language Understanding

> Query Embedding

> > OpenAI / sentence-transformers, Hugging Face , Customized Code


In [ ]:
#Lightweight opensource instruction LLM is used for embedding . This code doesn't focus on training!
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=128
)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
product_names = list(PRODUCTS.keys())
product_embeddings = embedder.encode(product_names)

print("\n\tSample one of the input product : \" ",product_names[1], "\"")
print("\n\tSize of the embedding returned for this product : ",product_embeddings[1].size)
print("\n\tSample first three features : ",product_embeddings[1][0], ":" , product_embeddings[1][1] ,":", product_embeddings[1][2])


#### Query Planner from Graph Search/Traversal

In [ ]:
#Note : Here LLM is used to convert NLP to Query seq plan!
def llm_query_planner(query):

    prompt = f"""
You are an e-commerce query planner.

Extract:

1. reference_product
2. constraints

Allowed constraints:
- cheaper
- more_durable
- more_comfortable

Return ONLY JSON.

Query:
{query}
"""
    output = llm(prompt)[0]["generated_text"]
    print("\n=========== RAW LLM QUERY PLAN ===========\n")
    print(output)
    # --------------------------------------------------------
    # SIMPLE JSON EXTRACTION
    # --------------------------------------------------------
    try:
        json_match = re.search(r"\{.*\}",output,re.DOTALL)
        parsed = json.loads(json_match.group())
        return parsed
    except:
        # fallback parser
        parsed = {"reference_product": None,"constraints": []}

        for product in PRODUCTS.keys():
            if product.lower() in query.lower():
                parsed["reference_product"] = product

        if "cheaper" in query.lower():
            parsed["constraints"].append("cheaper")
        if "durable" in query.lower():
            parsed["constraints"].append("more_durable")
        if "comfortable" in query.lower():
            parsed["constraints"].append("more_comfortable")
        return parsed

#TEST
print(llm_query_planner("shoes like Nike Air Max but cheaper and more durable"))


### Helper Functions : GEC | Entity Protection | etc.,

In [ ]:
#To check with product catalog to match the queried entity

def ground_reference_product(reference_text):
    query_embedding = embedder.encode([reference_text])[0]
    scores = []
    print("Query Embedding Subset: ",query_embedding[0],",",query_embedding[1],",",query_embedding[2])
    for i, emb in enumerate(product_embeddings):
        similarity = np.dot(query_embedding, emb)
        scores.append((product_names[i], similarity))
        print("\nProduct Compared: ",product_names[i],"Similarity Score Computed:",similarity)
        print("\tProduct Embedding Subset: ",product_embeddings[i][0],",",product_embeddings[i][1],",",product_embeddings[i][2])

    scores.sort(key=lambda x: x[1],reverse=True)
    return scores[0][0]

#TEST
print("\nInput Query : shoes like Nike Air Max but cheaper and more durable")
print("\nBest Match :",ground_reference_product("shoes like Nike but cheaper and more durable"))

## Retriever
> Single path Semantic Search or Orchested Search

> > Vector Search | Graph Traversal(Query) | ExternalAPI

> > Vector Database ((Pinecone / Weaviate / FAISS) |  Graph Database (Neo4J) Cypher / Gremlin | weather/ Stock | Customized Code


In [ ]:
def retrieve_graph_neighbors(product):
    neighbors = list(G.neighbors(product))
    return neighbors

#TEST
print("\nInput Product/Entity :ASICS Gel Nimbus")
print("\nNeighbors :",retrieve_graph_neighbors("ASICS Gel Nimbus"))

#### Graph Traversal with condition check
> > Multi-Hop reasoning

> > Filter

In [ ]:
def multi_hop_reasoning(reference_product,neighbors):
    ref_attrs = PRODUCTS[reference_product]
    enriched_candidates = []

    for product in neighbors:
        attrs = PRODUCTS[product]
        enriched_candidates.append({"product": product,"brand": attrs["brand"],"category": attrs["category"],"price": attrs["price"],"durability": attrs["durability"],
            "comfort": attrs["comfort"],"inventory": attrs["inventory"],
            "price_difference":ref_attrs["price"]- attrs["price"],
            "durability_gain":attrs["durability"]- ref_attrs["durability"],
            "comfort_gain":attrs["comfort"]- ref_attrs["comfort"]
        })
    return enriched_candidates

In [ ]:
def apply_constraints(candidates,constraints):
    filtered = []
    for item in candidates:
        valid = True
        if "cheaper" in constraints:
            if item["price_difference"] <= 0:
                valid = False

        if "more_durable" in constraints:
            if item["durability_gain"] <= 0:
                valid = False

        if "more_comfortable" in constraints:
            if item["comfort_gain"] <= 0:
                valid = False

        # inventory check
        if item["inventory"] <= 0:
            valid = False

        if valid:
            # deterministic ranking score
            score = 0
            score = 1.0 +(item["price_difference"] / 100) + (item["durability_gain"] * 0.5) + (item["comfort_gain"] * 0.3)
            item["score"] = round(score, 2)
            filtered.append(item)

    filtered.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return filtered


## Generator
> LM | LLM | SLM
> Open AI GPT,LangChain, Meta Llama/LlamaIndex , Mistral AI, Customised


In [ ]:
# Used to only relate explanation for only the retreived products
def llm_synthesis(query,reference_product,recommendations):

    context = ""

    for r in recommendations:
        context += f"""
Product: {r['product']}
Brand: {r['brand']}
Price: {r['price']}
Durability: {r['durability']}
Comfort: {r['comfort']}
Score: {r['score']}
"""

    prompt = f"""
You are an e-commerce recommendation assistant.

STRICT RULES:
- Use ONLY products from context
- Do NOT invent products
- Explain why recommendations satisfy constraints
- Keep response concise

User Query:
{query}

Reference Product:
{reference_product}

Candidate Products:
{context}

Generate recommendation summary.
"""
    response = llm(prompt)[0]["generated_text"]

    return response

## Orchestrator
> In hybrid system this works immediately
> > after the preprocessing
> > > using classifier (LM) to determine the most appropriate RAG to call
> > > to trigger multisource retreival

> > after the response generation
> > > to Combine results --> Reason --> Iterate (if needed)

> Microsoft Orchestration | Customized  

#### Language Model for responses

In [ ]:
!pip install transformers accelerate torch

In [ ]:
def graph_rag_pipeline(query):

    print("\n================================================")
    print("STEP 1 — LLM QUERY PLANNER")
    print("================================================")

    plan = llm_query_planner(query)

    pprint(plan)

    print("\n================================================")
    print("STEP 2 — ENTITY GROUNDING")
    print("================================================")

    grounded_product = ground_reference_product(
        plan["reference_product"]
    )

    print("Grounded Product:", grounded_product)

    print("\n================================================")
    print("STEP 3 — GRAPH RETRIEVAL")
    print("================================================")

    neighbors = retrieve_graph_neighbors(
        grounded_product
    )

    pprint(neighbors)

    print("\n================================================")
    print("STEP 4 — MULTI-HOP REASONING")
    print("================================================")

    candidates = multi_hop_reasoning(
        grounded_product,
        neighbors
    )

    pprint(candidates)

    print("\n================================================")
    print("STEP 5 — CONSTRAINT PROPAGATION")
    print("================================================")

    filtered = apply_constraints(
        candidates,
        plan["constraints"]
    )

    pprint(filtered)

    print("\n================================================")
    print("STEP 6 — LLM SYNTHESIS")
    print("================================================")

    final_response = llm_synthesis(
        query,
        grounded_product,
        filtered
    )

    print(final_response)

    return {"query_plan": plan,"grounded_product": grounded_product,"recommendations": filtered,"response": final_response}

In [ ]:
# ============================================================
# TEST QUERY
# ============================================================

query = (
    "shoes like Nike Air Max "
    "but cheaper and more durable"
)

results = graph_rag_pipeline(query)

## Evaluator
> Detect Failure:
> > Decide which layer need this : Confidence calibration across relevant component in the pipeline

> > Entity Map Accuracy | Hallucination | Constraint Check | Catalog Complaince | Raking Quality | Graph path validity

> Score Confidence level based on these

In [ ]:
def evaluate_entity_grounding(query,grounded_product,expected_product):
    return {"correct": grounded_product == expected_product,"expected": expected_product,"predicted": grounded_product }

def evaluate_constraints(reference_product,recommendations, constraints):
    ref = PRODUCTS[reference_product]
    violations = []
    for item in recommendations:
        # cheaper check
        if "cheaper" in constraints:
            if item["price"] >= ref["price"]:
                violations.append({"product": item["product"],"violation": "not cheaper"})
        # durability check
        if "more_durable" in constraints:
            if item["durability"] <= ref["durability"]:
                violations.append({"product": item["product"],"violation": "not more durable"})
        # comfort check
        if "more_comfortable" in constraints:
            if item["comfort"] <= ref["comfort"]:
                violations.append({"product": item["product"],"violation": "not more comfortable"})
    return {"passed": len(violations) == 0,"violations": violations}

def detect_hallucinations(llm_response,retrieved_products):
    retrieved_names = [ p["product"] for p in retrieved_products]
    hallucinated = []
    for product in PRODUCTS.keys():
        if (product in llm_response and product not in retrieved_names):
            hallucinated.append(product)
    return {"hallucination_detected":len(hallucinated) > 0, "hallucinated_products":hallucinated}

def evaluate_catalog_compliance(recommendations):
    invalid_products = []
    for item in recommendations:
        name = item["product"]
        # existence check
        if name not in PRODUCTS:
            invalid_products.append({"product": name,"reason": "missing_from_catalog"})
            continue
        # inventory check
        if PRODUCTS[name]["inventory"] <= 0:
            invalid_products.append({"product": name,"reason": "out_of_stock"})
    return {"catalog_compliant":len(invalid_products) == 0,"issues": invalid_products}

def evaluate_graph_paths(reference_product,recommendations):
    invalid_paths = []
    for item in recommendations:
        product = item["product"]
        if not G.has_edge(reference_product,product):
            invalid_paths.append(product)
    return {"valid_paths":len(invalid_paths) == 0, "broken_relationships": invalid_paths}

def evaluate_ranking_quality(recommendations):
    scores = [r["score"] for r in recommendations ]
    descending = all(scores[i] >= scores[i+1]for i in range(len(scores)-1))
    return {"ranking_valid": descending,"scores": scores}

def compute_confidence(grounding_eval,constraint_eval,hallucination_eval,catalog_eval,graph_eval,ranking_eval):
    score = 1.0
    if not grounding_eval["correct"]:
        score -= 0.25
    if not constraint_eval["passed"]:
        score -= 0.30
    if hallucination_eval["hallucination_detected"]:
        score -= 0.30
    if not catalog_eval["catalog_compliant"]:
        score -= 0.20
    if not graph_eval["valid_paths"]:
        score -= 0.20
    if not ranking_eval["ranking_valid"]:
        score -= 0.10
    return max(score, 0.0)

In [ ]:
def evaluate_graph_rag_system(query,expected_product,pipeline_output):

    grounding_eval = evaluate_entity_grounding(query,pipeline_output["grounded_product"], expected_product)
    constraint_eval = evaluate_constraints(pipeline_output["grounded_product"],pipeline_output["recommendations"],pipeline_output["query_plan"]["constraints"])
    hallucination_eval = detect_hallucinations(pipeline_output["response"],pipeline_output["recommendations"])
    catalog_eval = evaluate_catalog_compliance(pipeline_output["recommendations"])
    graph_eval = evaluate_graph_paths(pipeline_output["grounded_product"],pipeline_output["recommendations"])
    ranking_eval = evaluate_ranking_quality(pipeline_output["recommendations"])

    confidence = compute_confidence(grounding_eval,constraint_eval,hallucination_eval,catalog_eval, graph_eval,ranking_eval)

    return {"grounding_eval": grounding_eval,"constraint_eval": constraint_eval,"hallucination_eval": hallucination_eval,"catalog_eval": catalog_eval,
        "graph_eval": graph_eval, "ranking_eval": ranking_eval,"final_confidence": confidence }

## Human In the Loop Trigger (if needed)

In [ ]:
def human_review_required(confidence):
    if confidence < 0.6:
        return True
    return False


In [ ]:
user_input = "shoes like Nike Air Max but cheaper and more durable"

results = graph_rag_pipeline(user_input)
evaluation = evaluate_graph_rag_system(query=user_input,expected_product="Nike Air Max", pipeline_output=results)

#Human review
review_flag = human_review_required(evaluation['final_confidence'])

print("\n--- FINAL OUTPUT ---")
print("User Input:", user_input)
print("\n\tResponse:\n", results)
pprint(evaluation)
print("\n\tConfidence:", evaluation['final_confidence'])
print("\n\tHuman Review:", review_flag)